Step 1 — Check GPU

# Cross-Domain Hallucination Detection in Medical RAG

Pipeline:

PubMedQA
→ Medical Retrieval Corpus
→ Sentence Transformer
→ FAISS Retrieval
→ Medical RAG
→ LettuceDetect Baseline
→ Manual Span-Level Annotation
→ Medical Domain Adaptation
→ Same Held-Out Test Set
→ Precision / Recall / F1

In [1]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    device = "cuda"
else:
    device = "cpu"
    print("Using CPU")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


Step 2 — Install packages

In [2]:
!pip install -q \
    datasets \
    sentence-transformers \
    faiss-cpu \
    scikit-learn \
    tqdm \
    transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 71.5 MB/s eta 0:00:00:00:0100:01


Step 3 — Import and verify packages

In [3]:
import numpy as np
import pandas as pd
import torch
import faiss
import sklearn
import transformers
import sentence_transformers
import datasets

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Sentence Transformers:", sentence_transformers.__version__)
print("FAISS:", faiss.__version__)
print("Datasets:", datasets.__version__)
print("CUDA:", torch.cuda.is_available())

NumPy: 2.0.2
Pandas: 2.3.3
PyTorch: 2.10.0+cu128
Transformers: 5.0.0
Sentence Transformers: 5.4.1
FAISS: 1.15.1
Datasets: 5.0.0
CUDA: True


Step 4 — Load PubMedQA

In [5]:
from datasets import load_dataset

dataset = load_dataset(
    "qiaojin/PubMedQA",
    "pqa_labeled"
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['pubid', 'question', 'context', 'long_answer', 'final_decision'],
        num_rows: 1000
    })
})


Step 5 — Inspect the dataset

In [6]:
print(dataset.column_names)

example = dataset["train"][0]

for key, value in example.items():
    print("\n==============================")
    print(key)
    print("==============================")
    print(str(value)[:1500])

{'train': ['pubid', 'question', 'context', 'long_answer', 'final_decision']}

pubid
21645374

question
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

context
{'contexts': ['Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cells at the center of these areoles and progresses outwards, stopping approximately five cells from the vasculature. The role of mitochondria during PCD has been recognized in animals; however, it has been less studied during PCD in plants.', 'The following paper elucidates the role of mitochondrial dynamics during developmentally regulated PCD in vivo in A. madagascariensis. A single areole within a window stage leaf (PCD is occurring) was divided into three areas based on the p

Step 6 — Select all 1000 examples

In [7]:
N = 1000

medical_data = dataset["train"].select(
    range(min(N, len(dataset["train"])))
)

print("Number of examples:", len(medical_data))

Number of examples: 1000


Step 7 — Create the main DataFrame

In [8]:
medical_df = pd.DataFrame({
    "id": range(len(medical_data)),
    "pubid": medical_data["pubid"],
    "question": medical_data["question"],
    "context": medical_data["context"],
    "reference_answer": medical_data["long_answer"],
    "final_decision": medical_data["final_decision"]
})

print("DataFrame shape:", medical_df.shape)
print("Columns:", medical_df.columns.tolist())

medical_df.head()

DataFrame shape: (1000, 6)
Columns: ['id', 'pubid', 'question', 'context', 'reference_answer', 'final_decision']


,id,pubid,question,context,reference_answer,final_decision
0,0,21645374,Do mitochondria play a role in remodelling lac...,{'contexts': ['Programmed cell death (PCD) is ...,Results depicted mitochondrial dynamics in viv...,yes
1,1,16418930,Landolt C and snellen e acuity: differences in...,{'contexts': ['Assessment of visual acuity dep...,"Using the charts described, there was only a s...",no
2,2,9488747,"Syncope during bathing in infants, a pediatric...",{'contexts': ['Apparent life-threatening event...,"""Aquagenic maladies"" could be a pediatric form...",yes
3,3,17208539,Are the long-term results of the transanal pul...,{'contexts': ['The transanal endorectal pull-t...,Our long-term study showed significantly bette...,no
4,4,10808977,Can tailored interventions increase mammograph...,{'contexts': ['Telephone counseling and tailor...,The effects of the intervention were most pron...,yes


Step 8 — Split into retrieval corpus and held-out test set

In [9]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    medical_df,
    test_size=0.20,
    random_state=42,
    shuffle=True
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Total examples:", len(medical_df))
print("Training/retrieval examples:", len(train_df))
print("Held-out test examples:", len(test_df))

print("\nFirst 10 training IDs:")
print(train_df["id"].head(10).tolist())

print("\nFirst 10 test IDs:")
print(test_df["id"].head(10).tolist())

Total examples: 1000
Training/retrieval examples: 800
Held-out test examples: 200

First 10 training IDs:
[29, 535, 695, 557, 836, 596, 165, 918, 495, 824]

First 10 test IDs:
[521, 737, 740, 660, 411, 678, 626, 513, 859, 136]


Step 9 — Extract the real medical context

In [10]:
def extract_context(context):
    if isinstance(context, dict):
        contexts = context.get("contexts", [])

        if isinstance(contexts, list):
            return "\n".join(
                str(text) for text in contexts
            )

    return str(context)


train_df["context_text"] = train_df["context"].apply(
    extract_context
)

test_df["context_text"] = test_df["context"].apply(
    extract_context
)

print("Example extracted context:")
print(train_df["context_text"].iloc[0][:2000])

Example extracted context:
Measurement of visceral adipose tissue (VAT) needs to be accurate and sensitive to change for risk monitoring. The purpose of this study is to determine the CT slice location where VAT area can best reflect changes in VAT volume and body weight.
60 plain abdominal CT images from 30 males [mean age (range) 51 (41-68) years, mean body weight (range) 71.1 (101.9-50.9) kg] who underwent workplace screenings twice within a 1-year interval were evaluated. Automatically calculated and manually corrected areas of the VAT of various scan levels using "freeform curve" region of interest on CT were recorded and compared with body weight changes.
The strongest correlations of VAT area with VAT volume and body weight changes were shown in a slice 3 cm above the lower margin of L3 with r values of 0.853 and 0.902, respectively.


Step 10 — Load the embedding model

In [11]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2",
    device=device
)

print("Embedding model loaded successfully.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded successfully.


Step 11 — Create retrieval text

In [12]:
train_df["retrieval_text"] = (
    "Question: "
    + train_df["question"].astype(str)
    + "\nContext: "
    + train_df["context_text"].astype(str)
)

test_df["retrieval_text"] = (
    "Question: "
    + test_df["question"].astype(str)
    + "\nContext: "
    + test_df["context_text"].astype(str)
)

print(train_df["retrieval_text"].iloc[0][:3000])

Question: Visceral adipose tissue area measurement at a single level: can it represent visceral adipose tissue volume?
Context: Measurement of visceral adipose tissue (VAT) needs to be accurate and sensitive to change for risk monitoring. The purpose of this study is to determine the CT slice location where VAT area can best reflect changes in VAT volume and body weight.
60 plain abdominal CT images from 30 males [mean age (range) 51 (41-68) years, mean body weight (range) 71.1 (101.9-50.9) kg] who underwent workplace screenings twice within a 1-year interval were evaluated. Automatically calculated and manually corrected areas of the VAT of various scan levels using "freeform curve" region of interest on CT were recorded and compared with body weight changes.
The strongest correlations of VAT area with VAT volume and body weight changes were shown in a slice 3 cm above the lower margin of L3 with r values of 0.853 and 0.902, respectively.


Step 12 — Generate embeddings

In [13]:
train_embeddings = embedder.encode(
    train_df["retrieval_text"].tolist(),
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=16
).astype("float32")

print("Embedding shape:", train_embeddings.shape)

Batches:   0%|          | 0/50 [00:00<?, ?it/s]

Embedding shape: (800, 384)


Step 13 — Build FAISS index

In [14]:
train_dimension = train_embeddings.shape[1]

train_index = faiss.IndexFlatIP(
    train_dimension
)

train_index.add(train_embeddings)

print("Embedding dimension:", train_dimension)
print("Documents in FAISS index:", train_index.ntotal)

Embedding dimension: 384
Documents in FAISS index: 800


Step 14 — Create retrieval function

In [15]:
def retrieve_test_evidence(question, k=5):

    query_embedding = embedder.encode(
        [str(question)],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = train_index.search(
        query_embedding,
        k
    )

    results = train_df.iloc[indices[0]].copy()

    results["retrieval_score"] = scores[0]

    return results

Step 15 — Test retrieval on held-out questions

In [16]:
for test_number in range(5):

    question = test_df.iloc[test_number]["question"]

    retrieved = retrieve_test_evidence(
        question,
        k=5
    )

    print("\n" + "=" * 100)
    print("TEST QUESTION", test_number + 1)
    print("=" * 100)

    print(question)

    print("\nRetrieved documents:")

    for rank, (_, row) in enumerate(
        retrieved.iterrows(),
        start=1
    ):
        print(f"\nRank {rank}")
        print("ID:", row["id"])
        print("Score:", round(row["retrieval_score"], 4))
        print("Question:", row["question"])


TEST QUESTION 1
Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?

Retrieved documents:

Rank 1
ID: 428
Score: 0.5266
Question: Screening History Among Women with Invasive Cervical Cancer in an Academic Medical Center: Will We Miss Cancers Following Updated Guidelines?

Rank 2
ID: 243
Score: 0.4742
Question: Diffusion-weighted echo-planar MR imaging of primary parotid gland tumors: is a prediction of different histologic subtypes possible?

Rank 3
ID: 560
Score: 0.4609
Question: Can nonproliferative breast disease and proliferative breast disease without atypia be distinguished by fine-needle aspiration cytology?

Rank 4
ID: 37
Score: 0.4466
Question: Do follow-up recommendations for abnormal Papanicolaou smears influence patient adherence?

Rank 5
ID: 196
Score: 0.3895
Question: Does vaginal intraepithelial neoplasia have the same evolution as cervical intraepithelial neoplasia?

TEST QUESTION 2
Are patients with Werlhof's disease at increased ris

Step 16 — Test detailed retrieval

In [17]:
test_row = test_df.iloc[0]

question = test_row["question"]
test_id = test_row["id"]

retrieved = retrieve_test_evidence(
    question,
    k=20
)

print("=" * 100)
print("HELD-OUT TEST QUESTION")
print("=" * 100)

print("Test ID:", test_id)
print("Question:", question)

print("\n" + "=" * 100)
print("TOP 20 RETRIEVED QUESTIONS")
print("=" * 100)

for rank, (_, row) in enumerate(
    retrieved.iterrows(),
    start=1
):
    print(f"\nRank {rank}")
    print("ID:", row["id"])
    print("Score:", round(row["retrieval_score"], 4))
    print("Question:", row["question"])

HELD-OUT TEST QUESTION
Test ID: 521
Question: Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?

TOP 20 RETRIEVED QUESTIONS

Rank 1
ID: 428
Score: 0.5266
Question: Screening History Among Women with Invasive Cervical Cancer in an Academic Medical Center: Will We Miss Cancers Following Updated Guidelines?

Rank 2
ID: 243
Score: 0.4742
Question: Diffusion-weighted echo-planar MR imaging of primary parotid gland tumors: is a prediction of different histologic subtypes possible?

Rank 3
ID: 560
Score: 0.4609
Question: Can nonproliferative breast disease and proliferative breast disease without atypia be distinguished by fine-needle aspiration cytology?

Rank 4
ID: 37
Score: 0.4466
Question: Do follow-up recommendations for abnormal Papanicolaou smears influence patient adherence?

Rank 5
ID: 196
Score: 0.3895
Question: Does vaginal intraepithelial neoplasia have the same evolution as cervical intraepithelial neoplasia?

Rank 6
ID: 57
Score: 0.3759
Quest

Step 17 — Leakage check

In [18]:
retrieved_ids = retrieved["id"].tolist()

print("=" * 100)
print("LEAKAGE CHECK")
print("=" * 100)

print("Held-out test ID:", test_id)
print("Retrieved IDs:", retrieved_ids)

if test_id in retrieved_ids:
    print("\n❌ DATA LEAKAGE DETECTED")
else:
    print("\n✅ NO DATA LEAKAGE")

LEAKAGE CHECK
Held-out test ID: 521
Retrieved IDs: [428, 243, 560, 37, 196, 57, 170, 581, 962, 832, 564, 468, 291, 438, 933, 796, 991, 419, 368, 145]

✅ NO DATA LEAKAGE


Step 18 — Inspect actual retrieved evidence

In [19]:
for rank, (_, row) in enumerate(
    retrieved.head(5).iterrows(),
    start=1
):

    print("\n" + "=" * 100)
    print(f"RANK {rank}")
    print("=" * 100)

    print("ID:", row["id"])
    print(
        "Retrieval score:",
        round(row["retrieval_score"], 4)
    )

    print("Question:", row["question"])

    print("\nRetrieved context:")
    print(row["context_text"][:3000])


RANK 1
ID: 428
Retrieval score: 0.5266
Question: Screening History Among Women with Invasive Cervical Cancer in an Academic Medical Center: Will We Miss Cancers Following Updated Guidelines?

Retrieved context:
Updated guidelines for the screening and management of cervical cancer in the United States recommend starting Papanicolaou (Pap) testing at age 21 and screening less frequently with less aggressive management for abnormalities. We sought to examine updated Pap test screening guidelines and how they may affect the detection of invasive cervical cancer, especially among women<30 years of age.
Patients diagnosed at Brigham and Women's Hospital with invasive cervical cancer between 2002 and 2012 were retrospectively identified. Prior screening history was obtained and patients were divided into two groups based on age<30 years or age ≥30 years. The two groups were then compared with respect to demographics, pathological findings, and time to diagnosis.
A total of 288 patients with

**RETREIVAL IS BAD , SO WE TRY OTHER METHODS FOR STRONGER RETERIVAL PIPELINE**

Cell 21 — Install retrieval packages

In [20]:
!pip install -q rank_bm25

Cell 22 — Import BM25 and prepare passage corpus

In [21]:
from rank_bm25 import BM25Okapi
import re
import numpy as np
import pandas as pd

print("BM25 imported successfully")

BM25 imported successfully


Cell 23 — Convert the 800 training examples into passages

In [22]:
def build_passage_corpus(df):
    passages = []

    for _, row in df.iterrows():
        context = row["context"]

        if isinstance(context, dict):
            contexts = context.get("contexts", [])
        else:
            contexts = [str(context)]

        for passage_idx, passage in enumerate(contexts):
            passage = str(passage).strip()

            if not passage:
                continue

            passages.append({
                "source_id": int(row["id"]),
                "pubid": row["pubid"],
                "question": row["question"],
                "passage_id": passage_idx,
                "passage": passage,
                "reference_answer": row["reference_answer"],
                "final_decision": row["final_decision"]
            })

    return pd.DataFrame(passages)


passage_df = build_passage_corpus(train_df)

print("Number of training examples:", len(train_df))
print("Number of retrieval passages:", len(passage_df))
print()
print(passage_df.head(3)[
    ["source_id", "pubid", "passage_id", "passage"]
])

Number of training examples: 800
Number of retrieval passages: 2684

   source_id     pubid  passage_id  \
0         29  28707539           0   
1         29  28707539           1   
2         29  28707539           2   

                                             passage  
0  Measurement of visceral adipose tissue (VAT) n...  
1  60 plain abdominal CT images from 30 males [me...  
2  The strongest correlations of VAT area with VA...  


Cell 24 — Clean the passages for BM25

In [23]:
def tokenize_for_bm25(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return text.split()


bm25_corpus = [
    tokenize_for_bm25(text)
    for text in passage_df["passage"].tolist()
]

bm25 = BM25Okapi(bm25_corpus)

print("BM25 index built successfully")
print("Indexed passages:", len(bm25_corpus))

BM25 index built successfully
Indexed passages: 2684


Cell 25 — Create BM25 retrieval function

In [24]:
def retrieve_bm25(question, k=10):
    query_tokens = tokenize_for_bm25(question)

    scores = bm25.get_scores(query_tokens)

    top_indices = np.argsort(scores)[::-1][:k]

    results = passage_df.iloc[top_indices].copy()
    results["bm25_score"] = scores[top_indices]

    results = results.reset_index(drop=True)

    return results

Cell 26 — Test BM25 on your Pap-smear question

In [25]:
test_row = test_df.iloc[0]

question = test_row["question"]

print("TEST QUESTION:")
print(question)
print("\n" + "=" * 100)

bm25_results = retrieve_bm25(question, k=10)

for rank, (_, row) in enumerate(bm25_results.iterrows(), start=1):
    print(f"\nRANK {rank}")
    print(f"Source ID: {row['source_id']}")
    print(f"PubMed ID: {row['pubid']}")
    print(f"BM25 score: {row['bm25_score']:.4f}")
    print(f"Passage: {row['passage'][:1000]}")

TEST QUESTION:
Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?


RANK 1
Source ID: 37
PubMed ID: 10575390
BM25 score: 22.0339
Passage: To compare adherence to follow-up recommendations for colposcopy or repeated Papanicolaou (Pap) smears for women with previously abnormal Pap smear results.

RANK 2
Source ID: 37
PubMed ID: 10575390
BM25 score: 19.7111
Passage: All women with abnormal Pap smear results referred for initial colposcopy and a random sample of those referred for repeated Pap smear. Medical records were located and reviewed for 90 of 107 women referred for colposcopy and 153 of 225 women referred for repeated Pap smears.

RANK 3
Source ID: 428
PubMed ID: 26859535
BM25 score: 19.5955
Passage: Updated guidelines for the screening and management of cervical cancer in the United States recommend starting Papanicolaou (Pap) testing at age 21 and screening less frequently with less aggressive management for abnormalities. We sought to examine

MEDCPT

Cell 27 — Load biomedical MedCPT encoders

In [26]:
# Cell 27 — Load biomedical MedCPT encoders

from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

medcpt_query_name = "ncbi/MedCPT-Query-Encoder"
medcpt_article_name = "ncbi/MedCPT-Article-Encoder"

medcpt_query_tokenizer = AutoTokenizer.from_pretrained(
    medcpt_query_name
)

medcpt_query_model = AutoModel.from_pretrained(
    medcpt_query_name
).to(device)

medcpt_article_tokenizer = AutoTokenizer.from_pretrained(
    medcpt_article_name
)

medcpt_article_model = AutoModel.from_pretrained(
    medcpt_article_name
).to(device)

medcpt_query_model.eval()
medcpt_article_model.eval()

print("MedCPT models loaded successfully")
print("Device:", device)

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

... (7 outputs left out)

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MedCPT models loaded successfully
Device: cuda


Cell 28 — Create MedCPT passage text

In [27]:
# Cell 28 — Prepare text for MedCPT passage retrieval

passage_df["medcpt_text"] = (
    passage_df["question"].astype(str)
    + " "
    + passage_df["passage"].astype(str)
)

print("Prepared passages:", len(passage_df))
print()
print(passage_df[["source_id", "passage_id", "medcpt_text"]].head(2))

Prepared passages: 2684

   source_id  passage_id                                        medcpt_text
0         29           0  Visceral adipose tissue area measurement at a ...
1         29           1  Visceral adipose tissue area measurement at a ...


Cell 29 — Encode all 2,684 passages with MedCPT

In [28]:
# Cell 29 — Generate MedCPT passage embeddings

def encode_medcpt_articles(texts, batch_size=16):
    all_embeddings = []

    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]

        encoded = medcpt_article_tokenizer(
            batch,
            truncation=True,
            padding=True,
            return_tensors="pt",
            max_length=512
        )

        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }

        with torch.no_grad():
            outputs = medcpt_article_model(**encoded)
            embeddings = outputs.last_hidden_state[:, 0, :]

        embeddings = torch.nn.functional.normalize(
            embeddings,
            p=2,
            dim=1
        )

        all_embeddings.append(
            embeddings.cpu().numpy()
        )

        if start % (batch_size * 10) == 0:
            print(
                f"Encoded {min(start + batch_size, len(texts))}"
                f"/{len(texts)} passages"
            )

    return np.vstack(all_embeddings).astype("float32")


medcpt_passage_embeddings = encode_medcpt_articles(
    passage_df["medcpt_text"].tolist(),
    batch_size=16
)

print()
print("MedCPT embedding shape:", medcpt_passage_embeddings.shape)

Encoded 16/2684 passages
Encoded 176/2684 passages
Encoded 336/2684 passages
Encoded 496/2684 passages
Encoded 656/2684 passages
Encoded 816/2684 passages
Encoded 976/2684 passages
Encoded 1136/2684 passages
Encoded 1296/2684 passages
Encoded 1456/2684 passages
Encoded 1616/2684 passages
Encoded 1776/2684 passages
Encoded 1936/2684 passages
Encoded 2096/2684 passages
Encoded 2256/2684 passages
Encoded 2416/2684 passages
Encoded 2576/2684 passages

MedCPT embedding shape: (2684, 768)


Cell 30 — Build the MedCPT FAISS index

In [29]:
# Cell 30 — Build FAISS index for MedCPT

medcpt_dimension = medcpt_passage_embeddings.shape[1]

medcpt_index = faiss.IndexFlatIP(medcpt_dimension)

medcpt_index.add(medcpt_passage_embeddings)

print("MedCPT FAISS index built successfully")
print("Indexed passages:", medcpt_index.ntotal)
print("Embedding dimension:", medcpt_dimension)

MedCPT FAISS index built successfully
Indexed passages: 2684
Embedding dimension: 768


Cell 31 — Create MedCPT question retrieval function

In [30]:
# Cell 31 — MedCPT dense retrieval

def encode_medcpt_query(question):
    encoded = medcpt_query_tokenizer(
        [str(question)],
        truncation=True,
        padding=True,
        return_tensors="pt",
        max_length=64
    )

    encoded = {
        key: value.to(device)
        for key, value in encoded.items()
    }

    with torch.no_grad():
        outputs = medcpt_query_model(**encoded)
        embedding = outputs.last_hidden_state[:, 0, :]

    embedding = torch.nn.functional.normalize(
        embedding,
        p=2,
        dim=1
    )

    return embedding.cpu().numpy().astype("float32")


def retrieve_medcpt(question, k=10):

    query_embedding = encode_medcpt_query(question)

    scores, indices = medcpt_index.search(
        query_embedding,
        k
    )

    results = passage_df.iloc[indices[0]].copy()

    results["medcpt_score"] = scores[0]

    results = results.reset_index(drop=True)

    return results

Cell 32 — Test MedCPT on the same Pap-smear question

In [31]:
# Cell 32 — Test MedCPT retrieval

test_row = test_df.iloc[0]
question = test_row["question"]

print("TEST QUESTION:")
print(question)

print("\n" + "=" * 100)

medcpt_results = retrieve_medcpt(
    question,
    k=10
)

for rank, (_, row) in enumerate(
    medcpt_results.iterrows(),
    start=1
):
    print(f"\nRANK {rank}")
    print(f"Source ID: {row['source_id']}")
    print(f"PubMed ID: {row['pubid']}")
    print(f"MedCPT score: {row['medcpt_score']:.4f}")
    print(f"Passage: {row['passage'][:1000]}")

TEST QUESTION:
Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?


RANK 1
Source ID: 428
PubMed ID: 26859535
MedCPT score: 0.6290
Passage: Updated guidelines for the screening and management of cervical cancer in the United States recommend starting Papanicolaou (Pap) testing at age 21 and screening less frequently with less aggressive management for abnormalities. We sought to examine updated Pap test screening guidelines and how they may affect the detection of invasive cervical cancer, especially among women<30 years of age.

RANK 2
Source ID: 37
PubMed ID: 10575390
MedCPT score: 0.6266
Passage: All women with abnormal Pap smear results referred for initial colposcopy and a random sample of those referred for repeated Pap smear. Medical records were located and reviewed for 90 of 107 women referred for colposcopy and 153 of 225 women referred for repeated Pap smears.

RANK 3
Source ID: 37
PubMed ID: 10575390
MedCPT score: 0.6245
Passage: Routine 

Cell 33 — Create BM25 + MedCPT hybrid retrieval

In [36]:
# Cell 33 — Corrected Hybrid BM25 + MedCPT Retrieval

def retrieve_hybrid(question, k=10, candidate_k=50, rrf_k=60):

    # --------------------------------------------------
    # 1. Retrieve candidates from BM25
    # --------------------------------------------------
    bm25_results = retrieve_bm25(
        question,
        k=candidate_k
    ).copy()

    bm25_results["bm25_rank"] = np.arange(
        1,
        len(bm25_results) + 1
    )

    # --------------------------------------------------
    # 2. Retrieve candidates from MedCPT
    # --------------------------------------------------
    medcpt_results = retrieve_medcpt(
        question,
        k=candidate_k
    ).copy()

    medcpt_results["medcpt_rank"] = np.arange(
        1,
        len(medcpt_results) + 1
    )

    # --------------------------------------------------
    # 3. Create unique passage identifiers
    # --------------------------------------------------
    bm25_results["passage_key"] = (
        bm25_results["source_id"].astype(int).astype(str)
        + "_"
        + bm25_results["passage_id"].astype(int).astype(str)
    )

    medcpt_results["passage_key"] = (
        medcpt_results["source_id"].astype(int).astype(str)
        + "_"
        + medcpt_results["passage_id"].astype(int).astype(str)
    )

    # --------------------------------------------------
    # 4. Keep the UNION of candidates, but retain
    #    passage information from either retriever
    # --------------------------------------------------
    bm25_part = bm25_results[
        [
            "passage_key",
            "source_id",
            "pubid",
            "question",
            "passage_id",
            "passage",
            "bm25_score",
            "bm25_rank"
        ]
    ]

    medcpt_part = medcpt_results[
        [
            "passage_key",
            "medcpt_score",
            "medcpt_rank"
        ]
    ]

    combined = pd.merge(
        bm25_part,
        medcpt_part,
        on="passage_key",
        how="outer"
    )

    # --------------------------------------------------
    # 5. Remove any malformed candidates
    # --------------------------------------------------
    combined = combined.dropna(
        subset=[
            "source_id",
            "passage_id",
            "passage"
        ]
    ).copy()

    # --------------------------------------------------
    # 6. Reciprocal Rank Fusion
    #
    # IMPORTANT:
    # A document absent from one retriever receives
    # zero contribution from that retriever.
    # --------------------------------------------------
    combined["bm25_rrf"] = np.where(
        combined["bm25_rank"].notna(),
        1.0 / (rrf_k + combined["bm25_rank"]),
        0.0
    )

    combined["medcpt_rrf"] = np.where(
        combined["medcpt_rank"].notna(),
        1.0 / (rrf_k + combined["medcpt_rank"]),
        0.0
    )

    combined["rrf_score"] = (
        combined["bm25_rrf"]
        + combined["medcpt_rrf"]
    )

    # --------------------------------------------------
    # 7. Sort by hybrid score
    # --------------------------------------------------
    combined = combined.sort_values(
        "rrf_score",
        ascending=False
    ).reset_index(drop=True)

    # --------------------------------------------------
    # 8. Return top candidates
    # --------------------------------------------------
    return combined.head(k)

Cell 34 — Test the hybrid retriever

In [37]:
# Cell 34 — Test corrected hybrid retrieval

hybrid_results = retrieve_hybrid(
    question,
    k=20,
    candidate_k=50,
    rrf_k=60
)

print("TEST QUESTION:")
print(question)

print("\n" + "=" * 100)

for rank, (_, row) in enumerate(
    hybrid_results.iterrows(),
    start=1
):

    print(f"\nRANK {rank}")
    print(f"Source ID: {int(row['source_id'])}")
    print(f"PubMed ID: {int(row['pubid'])}")
    print(f"Passage ID: {int(row['passage_id'])}")

    print(
        f"BM25 score: "
        f"{row['bm25_score'] if pd.notna(row['bm25_score']) else 'N/A'}"
    )

    print(
        f"MedCPT score: "
        f"{row['medcpt_score'] if pd.notna(row['medcpt_score']) else 'N/A'}"
    )

    print(f"RRF score: {row['rrf_score']:.6f}")

    print(f"Passage: {row['passage'][:700]}")

TEST QUESTION:
Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?


RANK 1
Source ID: 428
PubMed ID: 26859535
Passage ID: 0
BM25 score: 19.59548791929272
MedCPT score: 0.628987193107605
RRF score: 0.032266
Passage: Updated guidelines for the screening and management of cervical cancer in the United States recommend starting Papanicolaou (Pap) testing at age 21 and screening less frequently with less aggressive management for abnormalities. We sought to examine updated Pap test screening guidelines and how they may affect the detection of invasive cervical cancer, especially among women<30 years of age.

RANK 2
Source ID: 37
PubMed ID: 10575390
Passage ID: 3
BM25 score: 19.711057572298444
MedCPT score: 0.626596212387085
RRF score: 0.032258
Passage: All women with abnormal Pap smear results referred for initial colposcopy and a random sample of those referred for repeated Pap smear. Medical records were located and reviewed for 90 of 107 women referred

Cell 35 — Check source diversity

In [38]:
# Cell 35 — Analyze source diversity

print("Hybrid source distribution:")
print()

source_counts = (
    hybrid_results["source_id"]
    .astype(int)
    .value_counts()
)

for source_id, count in source_counts.items():
    pubid = int(
        hybrid_results.loc[
            hybrid_results["source_id"] == source_id,
            "pubid"
        ].iloc[0]
    )

    print(
        f"Source {source_id} "
        f"(PubMed {pubid}): "
        f"{count} passage(s)"
    )

print()

print(
    "Unique source articles:",
    hybrid_results["source_id"].nunique()
)

print(
    "Retrieved passages:",
    len(hybrid_results)
)

Hybrid source distribution:

Source 37 (PubMed 10575390): 3 passage(s)
Source 428 (PubMed 26859535): 2 passage(s)
Source 862 (PubMed 19430778): 2 passage(s)
Source 962 (PubMed 20073599): 1 passage(s)
Source 138 (PubMed 22706226): 1 passage(s)
Source 560 (PubMed 9100537): 1 passage(s)
Source 693 (PubMed 21457946): 1 passage(s)
Source 457 (PubMed 27096199): 1 passage(s)
Source 768 (PubMed 24591144): 1 passage(s)
Source 532 (PubMed 15223779): 1 passage(s)
Source 189 (PubMed 17008699): 1 passage(s)
Source 446 (PubMed 9444542): 1 passage(s)
Source 426 (PubMed 10749257): 1 passage(s)
Source 145 (PubMed 9427037): 1 passage(s)
Source 271 (PubMed 25819796): 1 passage(s)
Source 362 (PubMed 18158048): 1 passage(s)

Unique source articles: 16
Retrieved passages: 20


Cell 36 — Select diverse final evidence

In [39]:
# Cell 36 — Diversity-aware final evidence selection

def select_diverse_evidence(
    hybrid_results,
    k=5,
    max_passages_per_source=2
):

    selected = []
    source_counts = {}

    for _, row in hybrid_results.iterrows():

        source_id = int(row["source_id"])

        current_count = source_counts.get(
            source_id,
            0
        )

        if current_count >= max_passages_per_source:
            continue

        selected.append(row)

        source_counts[source_id] = (
            current_count + 1
        )

        if len(selected) >= k:
            break

    return pd.DataFrame(selected).reset_index(drop=True)

Cell 37 — Generate our final retrieval result

In [40]:
# Cell 37 — Final hybrid evidence

final_evidence = select_diverse_evidence(
    hybrid_results,
    k=5,
    max_passages_per_source=2
)

print("FINAL RETRIEVED EVIDENCE")
print("=" * 100)

for rank, (_, row) in enumerate(
    final_evidence.iterrows(),
    start=1
):

    print(f"\nEVIDENCE {rank}")

    print(
        f"Source ID: {int(row['source_id'])}"
    )

    print(
        f"PubMed ID: {int(row['pubid'])}"
    )

    print(
        f"Passage ID: {int(row['passage_id'])}"
    )

    print(
        f"RRF score: {row['rrf_score']:.6f}"
    )

    print(
        f"Passage:\n{row['passage'][:1200]}"
    )

FINAL RETRIEVED EVIDENCE

EVIDENCE 1
Source ID: 428
PubMed ID: 26859535
Passage ID: 0
RRF score: 0.032266
Passage:
Updated guidelines for the screening and management of cervical cancer in the United States recommend starting Papanicolaou (Pap) testing at age 21 and screening less frequently with less aggressive management for abnormalities. We sought to examine updated Pap test screening guidelines and how they may affect the detection of invasive cervical cancer, especially among women<30 years of age.

EVIDENCE 2
Source ID: 37
PubMed ID: 10575390
Passage ID: 3
RRF score: 0.032258
Passage:
All women with abnormal Pap smear results referred for initial colposcopy and a random sample of those referred for repeated Pap smear. Medical records were located and reviewed for 90 of 107 women referred for colposcopy and 153 of 225 women referred for repeated Pap smears.

EVIDENCE 3
Source ID: 37
PubMed ID: 10575390
Passage ID: 0
RRF score: 0.032018
Passage:
To compare adherence to follow-up r

Cell 38 — Build article-level metadata

In [41]:
# Cell 38 — Build article-level metadata

article_df = (
    passage_df
    .groupby("source_id", as_index=False)
    .agg(
        pubid=("pubid", "first"),
        question=("question", "first"),
        num_passages=("passage_id", "count")
    )
)

print("Number of unique source articles:", len(article_df))
print()
print(article_df.head())

Number of unique source articles: 800

   source_id     pubid                                           question  \
0          0  21645374  Do mitochondria play a role in remodelling lac...   
1          1  16418930  Landolt C and snellen e acuity: differences in...   
2          2   9488747  Syncope during bathing in infants, a pediatric...   
3          3  17208539  Are the long-term results of the transanal pul...   
4          4  10808977  Can tailored interventions increase mammograph...   

   num_passages  
0             2  
1             3  
2             2  
3             3  
4             6  


Cell 39 — Retrieve more candidates before fusion

In [42]:
# Cell 39 — Retrieve larger candidate pools

def get_retrieval_candidates(
    question,
    candidate_k=100
):

    bm25_results = retrieve_bm25(
        question,
        k=candidate_k
    ).copy()

    medcpt_results = retrieve_medcpt(
        question,
        k=candidate_k
    ).copy()

    return bm25_results, medcpt_results

Cell 40 — Article-level hybrid ranking

In [47]:
# Cell 40 — Article-level hybrid ranking

def retrieve_article_hybrid(
    question,
    candidate_k=100,
    rrf_k=60,
    top_articles=10
):
    
    # ----------------------------------------------
    # Retrieve passage candidates
    # ----------------------------------------------
    
    bm25_results, medcpt_results = get_retrieval_candidates(
        question,
        candidate_k=candidate_k
    )
    
    # ----------------------------------------------
    # Convert BM25 passage results to article level
    # Keep the best passage from each article
    # ----------------------------------------------
    
    bm25_article = (
        bm25_results
        .groupby("source_id")
        .agg(
            bm25_best_score=("bm25_score", "max")
        )
        .reset_index()
    )
    
    bm25_article = (
        bm25_article
        .sort_values("bm25_best_score", ascending=False)
        .reset_index(drop=True)
    )
    
    bm25_article["bm25_rank"] = np.arange(len(bm25_article)) + 1
    
    # ----------------------------------------------
    # Convert MedCPT passage results to article level
    # Keep the best passage from each article
    # ----------------------------------------------
    
    medcpt_article = (
        medcpt_results
        .groupby("source_id")
        .agg(
            medcpt_best_score=("medcpt_score", "max")
        )
        .reset_index()
    )
    
    medcpt_article = (
        medcpt_article
        .sort_values("medcpt_best_score", ascending=False)
        .reset_index(drop=True)
    )
    
    medcpt_article["medcpt_rank"] = np.arange(len(medcpt_article)) + 1
    
    # ----------------------------------------------
    # Merge article rankings
    # ----------------------------------------------
    
    article_scores = pd.merge(
        bm25_article[
            [
                "source_id",
                "bm25_best_score",
                "bm25_rank"
            ]
        ],
        medcpt_article[
            [
                "source_id",
                "medcpt_best_score",
                "medcpt_rank"
            ]
        ],
        on="source_id",
        how="outer"
    )
    
    # ----------------------------------------------
    # Reciprocal Rank Fusion
    # ----------------------------------------------
    
    article_scores["bm25_rrf"] = np.where(
        article_scores["bm25_rank"].notna(),
        1.0 / (
            rrf_k + article_scores["bm25_rank"]
        ),
        0.0
    )
    
    article_scores["medcpt_rrf"] = np.where(
        article_scores["medcpt_rank"].notna(),
        1.0 / (
            rrf_k + article_scores["medcpt_rank"]
        ),
        0.0
    )
    
    article_scores["hybrid_score"] = (
        article_scores["bm25_rrf"]
        + article_scores["medcpt_rrf"]
    )
    
    # ----------------------------------------------
    # Add article metadata
    # ----------------------------------------------
    
    article_scores = article_scores.merge(
        article_df,
        on="source_id",
        how="left"
    )
    
    # ----------------------------------------------
    # Final article ranking
    # ----------------------------------------------
    
    article_scores = (
        article_scores
        .sort_values("hybrid_score", ascending=False)
        .reset_index(drop=True)
    )
    
    return article_scores.head(top_articles)

Cell 41 — Test article-level retrieval

In [48]:
# Cell 41 — Test article-level hybrid retrieval

article_results = retrieve_article_hybrid(
    question,
    candidate_k=100,
    rrf_k=60,
    top_articles=10
)

print("TEST QUESTION:")
print(question)

print("\n" + "=" * 100)

for rank, (_, row) in enumerate(
    article_results.iterrows(),
    start=1
):

    print(f"\nARTICLE RANK {rank}")

    print(
        f"Source ID: {int(row['source_id'])}"
    )

    print(
        f"PubMed ID: {int(row['pubid'])}"
    )

    print(
        f"Number of passages: "
        f"{int(row['num_passages'])}"
    )

    print(
        f"BM25 best score: "
        f"{row['bm25_best_score']}"
    )

    print(
        f"MedCPT best score: "
        f"{row['medcpt_best_score']}"
    )

    print(
        f"Hybrid score: "
        f"{row['hybrid_score']:.6f}"
    )

    print(
        f"Original question: "
        f"{row['question']}"
    )

TEST QUESTION:
Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?


ARTICLE RANK 1
Source ID: 37
PubMed ID: 10575390
Number of passages: 7
BM25 best score: 22.033908733474046
MedCPT best score: 0.626596212387085
Hybrid score: 0.032522
Original question: Do follow-up recommendations for abnormal Papanicolaou smears influence patient adherence?

ARTICLE RANK 2
Source ID: 428
PubMed ID: 26859535
Number of passages: 3
BM25 best score: 19.59548791929272
MedCPT best score: 0.628987193107605
Hybrid score: 0.032522
Original question: Screening History Among Women with Invasive Cervical Cancer in an Academic Medical Center: Will We Miss Cancers Following Updated Guidelines?

ARTICLE RANK 3
Source ID: 962
PubMed ID: 20073599
Number of passages: 4
BM25 best score: 14.682658344169592
MedCPT best score: 0.5833994746208191
Hybrid score: 0.031258
Original question: Do liquid-based preparations of urinary cytology perform differently than classically prepared cases?

 Cell 42 — Select the best passages from the top retrieved articles

In [57]:
# Cell 42 — Final evidence selection

def select_final_evidence(
    question,
    article_results,
    top_passages=5,
    passages_per_article=2
):
    
    selected_passages = []
    
    # Encode question once
    query_embedding = encode_medcpt_query(question)
    
    for _, article in article_results.iterrows():
        
        source_id = int(article["source_id"])
        article_hybrid_score = float(article["hybrid_score"])
        
        # Get all passages belonging to this article
        article_passages = passage_df[
            passage_df["source_id"] == source_id
        ].copy()
        
        if len(article_passages) == 0:
            continue
        
        # Prepare passage text
        passage_texts = (
            article_passages["question"].astype(str)
            + " "
            + article_passages["passage"].astype(str)
        ).tolist()
        
        # Encode passages with MedCPT
        encoded = medcpt_article_tokenizer(
            passage_texts,
            truncation=True,
            padding=True,
            return_tensors="pt",
            max_length=512
        )
        
        encoded = {
            key: value.to(device)
            for key, value in encoded.items()
        }
        
        with torch.no_grad():
            outputs = medcpt_article_model(**encoded)
            passage_embeddings = outputs.last_hidden_state[:, 0, :]
        
        passage_embeddings = torch.nn.functional.normalize(
            passage_embeddings,
            p=2,
            dim=1
        )
        
        passage_embeddings = passage_embeddings.cpu().numpy()
        
        # Question-passage semantic similarity
        passage_scores = np.dot(
            passage_embeddings,
            query_embedding[0]
        )
        
        article_passages["passage_medcpt_score"] = passage_scores
        
        # Keep only the strongest passages from this article
        article_passages = (
            article_passages
            .sort_values(
                "passage_medcpt_score",
                ascending=False
            )
            .head(passages_per_article)
        )
        
        # Preserve article-level retrieval score
        article_passages["article_hybrid_score"] = (
            article_hybrid_score
        )
        
        selected_passages.append(article_passages)
    
    if not selected_passages:
        return pd.DataFrame()
    
    selected_passages = pd.concat(
        selected_passages,
        ignore_index=True
    )
    
    # ------------------------------------------------
    # Normalize article-level scores
    # ------------------------------------------------
    
    article_min = selected_passages[
        "article_hybrid_score"
    ].min()
    
    article_max = selected_passages[
        "article_hybrid_score"
    ].max()
    
    if article_max > article_min:
        selected_passages["article_score_norm"] = (
            (
                selected_passages["article_hybrid_score"]
                - article_min
            )
            /
            (
                article_max
                - article_min
            )
        )
    else:
        selected_passages["article_score_norm"] = 1.0
    
    # ------------------------------------------------
    # Combined evidence score
    # ------------------------------------------------
    
    selected_passages["evidence_score"] = (
        0.7 * selected_passages["passage_medcpt_score"]
        +
        0.3 * selected_passages["article_score_norm"]
    )
    
    # ------------------------------------------------
    # Rank globally
    # ------------------------------------------------
    
    selected_passages = (
        selected_passages
        .sort_values(
            "evidence_score",
            ascending=False
        )
        .reset_index(drop=True)
    )
    
    # ------------------------------------------------
    # Limit number of passages from same article
    # ------------------------------------------------
    
    final_passages = []
    source_counts = {}
    
    for _, row in selected_passages.iterrows():
        
        source_id = int(row["source_id"])
        
        count = source_counts.get(source_id, 0)
        
        if count >= 2:
            continue
        
        final_passages.append(row)
        source_counts[source_id] = count + 1
        
        if len(final_passages) >= top_passages:
            break
    
    return pd.DataFrame(final_passages).reset_index(drop=True)

Cell 43 — Inspect retrieved evidenc

In [58]:
# Cell 43 — Inspect final retrieved evidence

retrieved_evidence = select_final_evidence(
    question,
    article_results,
    top_passages=5,
    passages_per_article=2
)

print("TEST QUESTION:")
print(question)

print("\n" + "=" * 100)

for rank, (_, row) in enumerate(
    retrieved_evidence.iterrows(),
    start=1
):
    
    print(f"\nFINAL EVIDENCE RANK {rank}")
    
    print(
        f"Source ID: {int(row['source_id'])}"
    )
    
    print(
        f"PubMed ID: {int(row['pubid'])}"
    )
    
    print(
        f"Passage ID: {int(row['passage_id'])}"
    )
    
    print(
        f"Article hybrid score: "
        f"{row['article_hybrid_score']:.6f}"
    )
    
    print(
        f"Passage MedCPT score: "
        f"{row['passage_medcpt_score']:.4f}"
    )
    
    print(
        f"Final evidence score: "
        f"{row['evidence_score']:.4f}"
    )
    
    print("\nSource article question:")
    print(row["question"])
    
    print("\nRetrieved passage:")
    print(row["passage"])
    
    print("\n" + "-" * 100)

TEST QUESTION:
Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?


FINAL EVIDENCE RANK 1
Source ID: 428
PubMed ID: 26859535
Passage ID: 0
Article hybrid score: 0.032522
Passage MedCPT score: 0.6290
Final evidence score: 0.7403

Source article question:
Screening History Among Women with Invasive Cervical Cancer in an Academic Medical Center: Will We Miss Cancers Following Updated Guidelines?

Retrieved passage:
Updated guidelines for the screening and management of cervical cancer in the United States recommend starting Papanicolaou (Pap) testing at age 21 and screening less frequently with less aggressive management for abnormalities. We sought to examine updated Pap test screening guidelines and how they may affect the detection of invasive cervical cancer, especially among women<30 years of age.

----------------------------------------------------------------------------------------------------

FINAL EVIDENCE RANK 2
Source ID: 37
PubMed ID: 105

Cell 44 — Run retrieval on the held-out test set

In [59]:
# Cell 44 — Run retrieval on the held-out test set

retrieval_results = []

for idx, row in test_df.iterrows():
    
    q = str(row["question"])
    
    articles = retrieve_article_hybrid(
        q,
        candidate_k=100,
        rrf_k=60,
        top_articles=10
    )
    
    evidence = select_final_evidence(
        q,
        articles,
        top_passages=5,
        passages_per_article=2
    )
    
    retrieval_results.append({
        "test_id": int(row["id"]),
        "question": q,
        "num_articles": len(articles),
        "num_evidence": len(evidence),
        "retrieved_source_ids": (
            evidence["source_id"].astype(int).tolist()
            if len(evidence) > 0
            else []
        ),
        "retrieved_pubids": (
            evidence["pubid"].astype(int).tolist()
            if len(evidence) > 0
            else []
        )
    })

retrieval_eval_df = pd.DataFrame(retrieval_results)

print("Retrieval evaluation completed.")
print(f"Test questions evaluated: {len(retrieval_eval_df)}")

retrieval_eval_df.head()

Retrieval evaluation completed.
Test questions evaluated: 200


,test_id,question,num_articles,num_evidence,retrieved_source_ids,retrieved_pubids
0,521,Pap smears with glandular cell abnormalities: ...,10,5,"[428, 37, 37, 428, 962]","[26859535, 10575390, 10575390, 26859535, 20073..."
1,737,Are patients with Werlhof's disease at increas...,10,5,"[348, 342, 348, 342, 930]","[12963175, 15141797, 12963175, 15141797, 21276..."
2,740,Is there a connection between sublingual varic...,10,5,"[834, 834, 840, 840, 160]","[26923375, 26923375, 26063028, 26063028, 15223..."
3,660,Does health information exchange reduce redund...,10,5,"[418, 333, 333, 418, 788]","[25501465, 21889895, 21889895, 25501465, 15774..."
4,411,The promise of specialty pharmaceuticals: are ...,10,5,"[206, 206, 407, 407, 458]","[26370095, 26370095, 17076590, 17076590, 22519..."


Cell 45 — Inspect retrieval source diversity

In [60]:
# Cell 45 — Inspect retrieval source diversity

all_sources = []

for sources in retrieval_eval_df["retrieved_source_ids"]:
    all_sources.extend(sources)

print(
    "Total retrieved evidence passages:",
    len(all_sources)
)

print(
    "Unique source articles retrieved:",
    len(set(all_sources))
)

print(
    "Average evidence passages per question:",
    retrieval_eval_df["num_evidence"].mean()
)

print("\nExample retrieved questions:")

for _, row in retrieval_eval_df.head(10).iterrows():
    
    print("\n" + "=" * 100)
    print("Test ID:", row["test_id"])
    print("Question:", row["question"])
    print("Retrieved sources:", row["retrieved_source_ids"])

Total retrieved evidence passages: 1000
Unique source articles retrieved: 381
Average evidence passages per question: 5.0

Example retrieved questions:

Test ID: 521
Question: Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?
Retrieved sources: [428, 37, 37, 428, 962]

Test ID: 737
Question: Are patients with Werlhof's disease at increased risk for bleeding complications when undergoing cardiac surgery?
Retrieved sources: [348, 342, 348, 342, 930]

Test ID: 740
Question: Is there a connection between sublingual varices and hypertension?
Retrieved sources: [834, 834, 840, 840, 160]

Test ID: 660
Question: Does health information exchange reduce redundant imaging?
Retrieved sources: [418, 333, 333, 418, 788]

Test ID: 411
Question: The promise of specialty pharmaceuticals: are they worth the price?
Retrieved sources: [206, 206, 407, 407, 458]

Test ID: 678
Question: The colour of pain: can patients use colour to describe osteoarthritis pain?
Retrieved

evaluate retrieval relevance

Cell 46 — Prepare manual retrieval evaluation set

In [64]:
# Cell 46 — Prepare manual retrieval evaluation set

# Use a fixed subset of held-out questions for manual relevance evaluation
retrieval_eval_test = test_df.sample(
    n=40,
    random_state=42
).reset_index(drop=True)

manual_retrieval_rows = []

for eval_idx, row in retrieval_eval_test.iterrows():
    question = str(row["question"])

    # BM25 top 5
    bm25_results = retrieve_bm25(
        question,
        k=5
    ).copy()

    for rank, (_, result) in enumerate(bm25_results.iterrows(), start=1):
        manual_retrieval_rows.append({
            "eval_id": eval_idx,
            "test_id": int(row["id"]),
            "question": question,
            "method": "BM25",
            "rank": rank,
            "source_id": int(result["source_id"]),
            "pubid": int(result["pubid"]),
            "passage_id": int(result["passage_id"]),
            "passage": str(result["passage"]),
            "relevance": np.nan
        })

    # MedCPT top 5
    medcpt_results = retrieve_medcpt(
        question,
        k=5
    ).copy()

    for rank, (_, result) in enumerate(medcpt_results.iterrows(), start=1):
        manual_retrieval_rows.append({
            "eval_id": eval_idx,
            "test_id": int(row["id"]),
            "question": question,
            "method": "MedCPT",
            "rank": rank,
            "source_id": int(result["source_id"]),
            "pubid": int(result["pubid"]),
            "passage_id": int(result["passage_id"]),
            "passage": str(result["passage"]),
            "relevance": np.nan
        })

    # Hybrid top 5
    article_results = retrieve_article_hybrid(
        question,
        candidate_k=100,
        rrf_k=60,
        top_articles=10
    )

    hybrid_results = select_final_evidence(
        question,
        article_results,
        top_passages=5,
        passages_per_article=2
    ).copy()

    for rank, (_, result) in enumerate(hybrid_results.iterrows(), start=1):
        manual_retrieval_rows.append({
            "eval_id": eval_idx,
            "test_id": int(row["id"]),
            "question": question,
            "method": "Hybrid",
            "rank": rank,
            "source_id": int(result["source_id"]),
            "pubid": int(result["pubid"]),
            "passage_id": int(result["passage_id"]),
            "passage": str(result["passage"]),
            "relevance": np.nan
        })

manual_retrieval_df = pd.DataFrame(manual_retrieval_rows)

print("Evaluation questions:", retrieval_eval_test.shape[0])
print("Total passages to manually evaluate:", manual_retrieval_df.shape[0])
print()
print(manual_retrieval_df["method"].value_counts())

Evaluation questions: 40
Total passages to manually evaluate: 600

method
BM25      200
MedCPT    200
Hybrid    200
Name: count, dtype: int64


Cell 47 — Calculate lexical evidence overlap

In [74]:
# Cell 47 — Manual retrieval relevance annotation

print("""
MANUAL RETRIEVAL LABELS

For each retrieved passage, assign:

2 = Directly Relevant
    The passage contains evidence that directly helps answer the question.

1 = Partially Relevant
    The passage is medically related to the question but provides
    only indirect, incomplete, or limited evidence.

0 = Not Relevant
    The passage does not provide useful evidence for answering the question.

Do NOT judge whether the passage is true or false.
Judge only whether it is useful evidence for this QUESTION.
""")

# Create a copy for manual annotation
manual_annotation_df = manual_retrieval_df.copy()

# Save a CSV so the passages can be reviewed manually
manual_annotation_path = "/kaggle/working/manual_retrieval_annotation.csv"
manual_annotation_df.to_csv(
    manual_annotation_path,
    index=False
)

print("Annotation file created:")
print(manual_annotation_path)

print()
print("Rows to annotate:", len(manual_annotation_df))
print()
print("After annotation, fill the 'relevance' column with 0, 1, or 2.")


MANUAL RETRIEVAL LABELS

For each retrieved passage, assign:

2 = Directly Relevant
    The passage contains evidence that directly helps answer the question.

1 = Partially Relevant
    The passage is medically related to the question but provides
    only indirect, incomplete, or limited evidence.

0 = Not Relevant
    The passage does not provide useful evidence for answering the question.

Do NOT judge whether the passage is true or false.
Judge only whether it is useful evidence for this QUESTION.

Annotation file created:
/kaggle/working/manual_retrieval_annotation.csv

Rows to annotate: 600

After annotation, fill the 'relevance' column with 0, 1, or 2.


Cell 48 — Evaluate all 200 questions

In [80]:
# Cell 48 — Evaluate the 40-question retrieval sample

import pandas as pd
import numpy as np

# IMPORTANT:
# Change this path to the actual Kaggle path of your uploaded CSV.
annotated_path = "/kaggle/input/datasets/classnotes/manual-retrieval-annotation-labeled-csv/manual_retrieval_annotation_labeled.csv"

# Load the labeled retrieval data
annotated_retrieval_df = pd.read_csv(annotated_path)

# Make sure relevance is numeric
annotated_retrieval_df["relevance"] = pd.to_numeric(
    annotated_retrieval_df["relevance"],
    errors="coerce"
)

# Check annotation completeness
missing_labels = annotated_retrieval_df["relevance"].isna().sum()

print("Total annotation rows:", len(annotated_retrieval_df))
print("Missing relevance labels:", missing_labels)

print("\nRelevance label distribution:")
print(
    annotated_retrieval_df["relevance"]
    .value_counts()
    .sort_index()
)

# ---------------------------------------------------------
# Metric functions
# ---------------------------------------------------------

def precision_at_k(group, k=5):
    top_k = group.sort_values("rank").head(k)
    return (top_k["relevance"] > 0).mean()


def ndcg_at_k(group, k=5):
    top_k = group.sort_values("rank").head(k)
    relevance = top_k["relevance"].to_numpy(dtype=float)

    if len(relevance) == 0:
        return 0.0

    discounts = np.log2(np.arange(2, len(relevance) + 2))

    dcg = np.sum(
        (2 ** relevance - 1) / discounts
    )

    ideal_relevance = np.sort(relevance)[::-1]

    ideal_dcg = np.sum(
        (2 ** ideal_relevance - 1) / discounts
    )

    if ideal_dcg == 0:
        return 0.0

    return dcg / ideal_dcg


def reciprocal_rank(group):
    ranked = group.sort_values("rank")

    for _, row in ranked.iterrows():
        if row["relevance"] > 0:
            return 1.0 / float(row["rank"])

    return 0.0


# ---------------------------------------------------------
# Calculate metrics for each retrieval method
# ---------------------------------------------------------

retrieval_metric_rows = []

for method in ["BM25", "MedCPT", "Hybrid"]:

    method_df = annotated_retrieval_df[
        annotated_retrieval_df["method"] == method
    ].copy()

    grouped = method_df.groupby("eval_id")

    p5_values = []
    ndcg5_values = []
    rr_values = []

    for _, group in grouped:

        p5_values.append(
            precision_at_k(group, k=5)
        )

        ndcg5_values.append(
            ndcg_at_k(group, k=5)
        )

        rr_values.append(
            reciprocal_rank(group)
        )

    retrieval_metric_rows.append({
        "method": method,
        "questions": len(p5_values),
        "Precision@5": np.mean(p5_values),
        "nDCG@5": np.mean(ndcg5_values),
        "MRR": np.mean(rr_values)
    })


retrieval_metrics_df = pd.DataFrame(
    retrieval_metric_rows
)

print("\nRetrieval evaluation results:")
display(retrieval_metrics_df)

Total annotation rows: 600
Missing relevance labels: 0

Relevance label distribution:
relevance
0    259
1    296
2     45
Name: count, dtype: int64

Retrieval evaluation results:


,method,questions,Precision@5,nDCG@5,MRR
0,BM25,40,0.790,0.901287,0.906250
1,MedCPT,40,0.375,0.570038,0.557083
2,Hybrid,40,0.540,0.826582,0.835417


Cell 49 — Define the final BM25 evidence retrieval

In [81]:
# Cell 49 — Final BM25 evidence retrieval for RAG

def retrieve_final_evidence(question, top_passages=5):
    """
    Retrieve the top BM25 passages for a question.

    These passages will be provided to the RAG generator.
    """

    results = retrieve_bm25(
        question,
        k=top_passages
    ).copy()

    results = results.reset_index(drop=True)

    return results


# Test retrieval on the first held-out question
sample_question = str(test_df.iloc[0]["question"])

sample_evidence = retrieve_final_evidence(
    sample_question,
    top_passages=5
)

print("Question:")
print(sample_question)

print("\nRetrieved evidence:")
for i, row in sample_evidence.iterrows():
    print(f"\n--- Evidence {i + 1} ---")
    print(f"Source ID: {row['source_id']}")
    print(f"PubMed ID: {row['pubid']}")
    print(f"BM25 score: {row['bm25_score']:.4f}")
    print(row["passage"])

Question:
Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?

Retrieved evidence:

--- Evidence 1 ---
Source ID: 37
PubMed ID: 10575390
BM25 score: 22.0339
To compare adherence to follow-up recommendations for colposcopy or repeated Papanicolaou (Pap) smears for women with previously abnormal Pap smear results.

--- Evidence 2 ---
Source ID: 37
PubMed ID: 10575390
BM25 score: 19.7111
All women with abnormal Pap smear results referred for initial colposcopy and a random sample of those referred for repeated Pap smear. Medical records were located and reviewed for 90 of 107 women referred for colposcopy and 153 of 225 women referred for repeated Pap smears.

--- Evidence 3 ---
Source ID: 428
PubMed ID: 26859535
BM25 score: 19.5955
Updated guidelines for the screening and management of cervical cancer in the United States recommend starting Papanicolaou (Pap) testing at age 21 and screening less frequently with less aggressive management for abnormaliti

Cell 50 — Create the RAG prompt

In [92]:
# Cell 58 — Final grounded RAG prompt

def build_rag_prompt(question, evidence_df):
    evidence_blocks = []

    for i, row in evidence_df.iterrows():
        evidence_blocks.append(
            f"[Evidence {i + 1}]\n"
            f"{row['passage']}"
        )

    evidence_text = "\n\n".join(evidence_blocks)

    prompt = f"""
You are a medical question-answering assistant.

Answer the question using ONLY the retrieved evidence.

Rules:
1. Use only information explicitly supported by the retrieved evidence.
2. Do not use outside medical knowledge.
3. Do not invent facts, numbers, diagnoses, treatments, or conclusions.
4. If the evidence is insufficient to answer the question, say:
   "The available evidence is insufficient to answer this question."
5. When the evidence supports an answer, provide a concise answer based only on that evidence.

Question:
{question}

Retrieved Evidence:
{evidence_text}

Answer:
""".strip()

    return prompt

The available evidence is insufficient to answer this question.


Cell 51 — Check which generation models are available

In [84]:
# Cell 51 — Check GPU memory before loading the RAG generation model

import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

    print("GPU:", gpu_name)
    print(f"Total GPU memory: {gpu_memory:.2f} GB")
else:
    print("CUDA is not available.")

GPU: Tesla T4
Total GPU memory: 14.56 GB


Cell 52 — Load the RAG generation model

In [94]:
# Cell 52 — Load the RAG generation model

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

rag_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

rag_tokenizer = AutoTokenizer.from_pretrained(
    rag_model_name
)

rag_model = AutoModelForCausalLM.from_pretrained(
    rag_model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

rag_model.eval()

print("RAG model loaded successfully.")
print("Model:", rag_model_name)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

RAG model loaded successfully.
Model: Qwen/Qwen2.5-1.5B-Instruct


Cell 53 — Generate one RAG answer

In [95]:
# Cell 53 — Generate one sample RAG answer

def generate_rag_answer(
    question,
    evidence_df,
    max_new_tokens=200
):
    prompt = build_rag_prompt(
        question,
        evidence_df
    )

    inputs = rag_tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=4096
    )

    inputs = {
        key: value.to(rag_model.device)
        for key, value in inputs.items()
    }

    with torch.no_grad():
        outputs = rag_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            top_p=None,
            top_k=None,
            pad_token_id=rag_tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    answer = rag_tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    return answer

Cell 54 — Inspect the complete RAG result

In [96]:
# Cell 54 — Inspect question, evidence, reference answer, and generated answer

sample_test_row = test_df[
    test_df["id"] == int(test_df.iloc[0]["id"])
].iloc[0]

print("=" * 80)
print("QUESTION")
print("=" * 80)
print(sample_question)

print("\n")
print("=" * 80)
print("RETRIEVED EVIDENCE")
print("=" * 80)

for i, row in sample_evidence.iterrows():
    print(f"\n[Evidence {i + 1}]")
    print(f"Source ID: {row['source_id']}")
    print(f"PubMed ID: {row['pubid']}")
    print(row["passage"])

print("\n")
print("=" * 80)
print("REFERENCE ANSWER")
print("=" * 80)
print(sample_test_row["reference_answer"])

print("\n")
print("=" * 80)
print("GENERATED RAG ANSWER")
print("=" * 80)
print(sample_answer)

QUESTION
Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?


RETRIEVED EVIDENCE

[Evidence 1]
Source ID: 37
PubMed ID: 10575390
To compare adherence to follow-up recommendations for colposcopy or repeated Papanicolaou (Pap) smears for women with previously abnormal Pap smear results.

[Evidence 2]
Source ID: 37
PubMed ID: 10575390
All women with abnormal Pap smear results referred for initial colposcopy and a random sample of those referred for repeated Pap smear. Medical records were located and reviewed for 90 of 107 women referred for colposcopy and 153 of 225 women referred for repeated Pap smears.

[Evidence 3]
Source ID: 428
PubMed ID: 26859535
Updated guidelines for the screening and management of cervical cancer in the United States recommend starting Papanicolaou (Pap) testing at age 21 and screening less frequently with less aggressive management for abnormalities. We sought to examine updated Pap test screening guidelines and how they may

evaluate whether the problem is retrieval or generation

Cell 55 — Check whether the reference answer is present in retrieved evidence

In [97]:
# Cell 55 — Check whether retrieved evidence contains information from the reference answer

def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]+", " ", text)
    return text


def reference_overlap_with_evidence(reference_answer, evidence_df):
    reference_text = normalize_text(reference_answer)
    evidence_text = " ".join(
        normalize_text(passage)
        for passage in evidence_df["passage"].tolist()
    )

    reference_tokens = set(reference_text.split())
    evidence_tokens = set(evidence_text.split())

    if not reference_tokens:
        return 0.0

    overlap = reference_tokens.intersection(evidence_tokens)

    return len(overlap) / len(reference_tokens)


reference_overlap = reference_overlap_with_evidence(
    sample_test_row["reference_answer"],
    sample_evidence
)

print("Reference/evidence token overlap:", round(reference_overlap, 4))

if reference_overlap > 0:
    print("Some reference-answer terms are present in the retrieved evidence.")
else:
    print("No reference-answer terms were found in the retrieved evidence.")

Reference/evidence token overlap: 0.4103
Some reference-answer terms are present in the retrieved evidence.


Cell 56 — Compare BM25 top 5, 10, and 20

In [98]:
# Cell 56 — Inspect more BM25 results for the sample question

for k in [5, 10, 20]:
    results = retrieve_bm25(
        sample_question,
        k=k
    )

    print("\n" + "=" * 80)
    print(f"BM25 TOP {k}")
    print("=" * 80)

    for i, row in results.iterrows():
        print(f"\n[{i + 1}] Source ID: {row['source_id']} | PubMed ID: {row['pubid']}")
        print(f"BM25 score: {row['bm25_score']:.4f}")
        print(row["passage"])


BM25 TOP 5

[1] Source ID: 37 | PubMed ID: 10575390
BM25 score: 22.0339
To compare adherence to follow-up recommendations for colposcopy or repeated Papanicolaou (Pap) smears for women with previously abnormal Pap smear results.

[2] Source ID: 37 | PubMed ID: 10575390
BM25 score: 19.7111
All women with abnormal Pap smear results referred for initial colposcopy and a random sample of those referred for repeated Pap smear. Medical records were located and reviewed for 90 of 107 women referred for colposcopy and 153 of 225 women referred for repeated Pap smears.

[3] Source ID: 428 | PubMed ID: 26859535
BM25 score: 19.5955
Updated guidelines for the screening and management of cervical cancer in the United States recommend starting Papanicolaou (Pap) testing at age 21 and screening less frequently with less aggressive management for abnormalities. We sought to examine updated Pap test screening guidelines and how they may affect the detection of invasive cervical cancer, especially amon

Cell 57 — Search the entire retrieval corpus for the answer-bearing passage

In [99]:
# Cell 57 — Search the entire retrieval corpus for answer-bearing terms

search_terms = [
    "rapid prescreening",
    "rapid prescreen",
    "RPS",
    "glandular cell",
    "AGC",
    "36.4%"
]

for term in search_terms:
    matches = passage_df[
        passage_df["passage"].astype(str).str.contains(
            term,
            case=False,
            regex=False,
            na=False
        )
    ]

    print("\n" + "=" * 80)
    print(f"SEARCH TERM: {term}")
    print(f"MATCHES: {len(matches)}")
    print("=" * 80)

    for _, row in matches.head(10).iterrows():
        print(
            f"\nSource ID: {row['source_id']} | "
            f"PubMed ID: {row['pubid']} | "
            f"Passage ID: {row['passage_id']}"
        )
        print(row["passage"])


SEARCH TERM: rapid prescreening
MATCHES: 0

SEARCH TERM: rapid prescreen
MATCHES: 0

SEARCH TERM: RPS
MATCHES: 1

Source ID: 592 | PubMed ID: 23360491 | Passage ID: 1
Event-related potentials (ERPs) differed as a function of emotional prosody both when attended and when unattended. While emotional prosody effects interacted with effects of spatial attention at early processing levels (<200 ms), these effects were additive at later processing stages (>200 ms).

SEARCH TERM: glandular cell
MATCHES: 0

SEARCH TERM: AGC
MATCHES: 1

Source ID: 847 | PubMed ID: 25604390 | Passage ID: 2
Loss of DKK3 expression was found in 64 of 158 (40.5%) samples, and it was associated with advanced T stage (p<0.001), lymph node metastasis (p<0.001), UICC TNM stage (p<0.001), tumor location (p = 0.029), lymphovascular invasion (p = 0.035), and perineural invasion (p = 0.032). Patients without DKK3 expression in tumor cells had a significantly worse disease-free and overall survival than those with DKK3 exp

Cell 59 — Generate RAG answers for all 200 test questions

In [100]:
# Cell 59 — Generate RAG answers for all held-out test questions

rag_results = []

for idx, row in test_df.iterrows():

    question = str(row["question"])

    # Retrieve final BM25 evidence
    evidence = retrieve_final_evidence(
        question,
        top_passages=5
    )

    # Generate grounded RAG answer
    answer = generate_rag_answer(
        question,
        evidence,
        max_new_tokens=200
    )

    rag_results.append({
        "test_id": int(row["id"]),
        "question": question,
        "retrieved_source_ids": (
            evidence["source_id"].astype(int).tolist()
            if len(evidence) > 0 else []
        ),
        "retrieved_pubids": (
            evidence["pubid"].astype(int).tolist()
            if len(evidence) > 0 else []
        ),
        "retrieved_evidence": (
            evidence["passage"].astype(str).tolist()
            if len(evidence) > 0 else []
        ),
        "generated_answer": answer,
        "reference_answer": str(row["reference_answer"]),
        "final_decision": str(row["final_decision"])
    })

    if (idx + 1) % 10 == 0:
        print(f"Generated {idx + 1}/{len(test_df)} answers")

rag_results_df = pd.DataFrame(rag_results)

print("\nRAG generation completed.")
print("Number of test questions:", len(rag_results_df))

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Generated 10/200 answers
Generated 20/200 answers
Generated 30/200 answers
Generated 40/200 answers
Generated 50/200 answers
Generated 60/200 answers
Generated 70/200 answers
Generated 80/200 answers
Generated 90/200 answers
Generated 100/200 answers
Generated 110/200 answers
Generated 120/200 answers
Generated 130/200 answers
Generated 140/200 answers
Generated 150/200 answers
Generated 160/200 answers
Generated 170/200 answers
Generated 180/200 answers
Generated 190/200 answers
Generated 200/200 answers

RAG generation completed.
Number of test questions: 200


Cell 60 — Inspect the RAG dataset

In [101]:
# Cell 60 — Inspect generated RAG results

print("Shape:", rag_results_df.shape)
print("\nColumns:")
print(rag_results_df.columns.tolist())

print("\nFirst 3 questions and generated answers:")

for i, row in rag_results_df.head(3).iterrows():
    print("\n" + "=" * 80)
    print("TEST ID:", row["test_id"])
    print("QUESTION:", row["question"])
    print("\nGENERATED ANSWER:")
    print(row["generated_answer"])

Shape: (200, 8)

Columns:
['test_id', 'question', 'retrieved_source_ids', 'retrieved_pubids', 'retrieved_evidence', 'generated_answer', 'reference_answer', 'final_decision']

First 3 questions and generated answers:

TEST ID: 521
QUESTION: Pap smears with glandular cell abnormalities: Are they detected by rapid prescreening?

GENERATED ANSWER:
The available evidence is insufficient to answer this question.

TEST ID: 737
QUESTION: Are patients with Werlhof's disease at increased risk for bleeding complications when undergoing cardiac surgery?

GENERATED ANSWER:
Based on the provided evidence, there is no direct mention of Werlhof's disease specifically or its association with bleeding complications during cardiac surgery. Therefore, the available evidence is insufficient to answer this question.

TEST ID: 740
QUESTION: Is there a connection between sublingual varices and hypertension?

GENERATED ANSWER:
Based on the provided evidence, there appears to be no direct established connection

Cell 61 — Count insufficient-evidence answers

In [102]:
# Cell 61 — Analyze insufficient-evidence responses

insufficient_phrase = (
    "available evidence is insufficient"
)

rag_results_df["insufficient_evidence"] = (
    rag_results_df["generated_answer"]
    .astype(str)
    .str.lower()
    .str.contains(insufficient_phrase, regex=False)
)

print(
    "Insufficient-evidence answers:",
    int(rag_results_df["insufficient_evidence"].sum())
)

print(
    "Answers with generated content:",
    int((~rag_results_df["insufficient_evidence"]).sum())
)

print(
    "Total answers:",
    len(rag_results_df)
)

Insufficient-evidence answers: 147
Answers with generated content: 53
Total answers: 200


Cell 62 — Save the RAG results

In [103]:
# Cell 62 — Save the complete RAG results

rag_output_path = "/kaggle/working/medical_rag_results.csv"

rag_results_df.to_csv(
    rag_output_path,
    index=False
)

print("Saved RAG results to:")
print(rag_output_path)

Saved RAG results to:
/kaggle/working/medical_rag_results.csv


In [108]:
import shutil
import os

source_file = "/kaggle/working/medical_rag_results.csv"
backup_file = "/kaggle/working/medical_rag_results_notebook1_final.csv"

shutil.copy(source_file, backup_file)

print("Saved:", backup_file)
print("File size:", round(os.path.getsize(backup_file) / 1024, 2), "KB")

Saved: /kaggle/working/medical_rag_results_notebook1_final.csv
File size: 639.19 KB


Cell 64 — Verify the file

In [109]:
import pandas as pd

final_check = pd.read_csv(
    "/kaggle/working/medical_rag_results_notebook1_final.csv"
)

print("Shape:", final_check.shape)
print("Columns:")
print(final_check.columns.tolist())

display(final_check.head(3))

Shape: (200, 9)
Columns:
['test_id', 'question', 'retrieved_source_ids', 'retrieved_pubids', 'retrieved_evidence', 'generated_answer', 'reference_answer', 'final_decision', 'insufficient_evidence']


,test_id,question,retrieved_source_ids,retrieved_pubids,retrieved_evidence,generated_answer,reference_answer,final_decision,insufficient_evidence
0,521,Pap smears with glandular cell abnormalities: ...,"[37, 37, 428, 962, 693]","[10575390, 10575390, 26859535, 20073599, 21457...",['To compare adherence to follow-up recommenda...,The available evidence is insufficient to answ...,Pap smears with glandular cell abnormalities a...,yes,True
1,737,Are patients with Werlhof's disease at increas...,"[348, 176, 665, 446, 342]","[12963175, 25156467, 24073931, 9444542, 15141797]",['It was the aim of the present study to elabo...,"Based on the provided evidence, there is no di...",Patients with WD may possibly undergo cardiac ...,no,True
2,740,Is there a connection between sublingual varic...,"[834, 915, 392, 160, 470]","[26923375, 22768311, 16319544, 15223725, 27050...",['Esophageal varices are present in 30% to 40%...,"Based on the provided evidence, there appears ...",An association was found between sublingual va...,yes,False
